# Person 1 Colab GPU Training Notebook (Data, Tokenizers, Baseline & Legal-BERT)


In [1]:
# Step 0: Check Colab Pro GPU & Install Dependencies
!nvidia-smi
!pip install -q datasets transformers accelerate scikit-learn pandas numpy tqdm

Fri Oct 16 19:48:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Step 1: Load LEDGAR Dataset & Compute Class Imbalance Table
import json
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
label_names = ledgar["train"].features["label"].names
num_classes = len(label_names)
print(ledgar)

train_counts = np.bincount(ledgar["train"]["label"], minlength=num_classes)
val_counts = np.bincount(ledgar["validation"]["label"], minlength=num_classes)
test_counts = np.bincount(ledgar["test"]["label"], minlength=num_classes)

class_df = pd.DataFrame([
    {
        "class_id": i,
        "class_name": name,
        "train_count": int(train_counts[i]),
        "val_count": int(val_counts[i]),
        "test_count": int(test_counts[i]),
    }
    for i, name in enumerate(label_names)
]).sort_values("train_count", ascending=False).reset_index(drop=True)

class_df.to_csv(OUTPUTS_DIR / "class_distribution.csv", index=False)
print("Most frequent class:", class_df.iloc[0].to_dict())
print("Least frequent class:", class_df.iloc[-1].to_dict())
print("Number of classes with < 10 test examples:", int((class_df["test_count"] < 10).sum()))

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

ledgar/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

ledgar/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 10000
    })
})
Most frequent class: {'class_id': 47, 'class_name': 'Governing Laws', 'train_count': 3167, 'val_count': 494, 'test_count': 582}
Least frequent class: {'class_id': 14, 'class_name': 'Books', 'train_count': 23, 'val_count': 0, 'test_count': 2}
Number of classes with < 10 test examples: 3


In [3]:
# Step 2: Compare Tokenizers (Legal-BERT vs. BERT-base vs. CamemBERT)
# Proves why we use Legal-BERT instead of CamemBERT on English SEC contracts
sample_texts = list(ledgar["train"]["text"][:500])
word_counts = np.array([max(1, len(t.split())) for t in sample_texts])

tokenizer_stats = {}
for model_name in ["nlpaueb/legal-bert-base-uncased", "bert-base-uncased", "camembert-base"]:
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = tok(sample_texts, truncation=False)
    tok_lens = np.array([len(ids) for ids in enc["input_ids"]])
    tokenizer_stats[model_name] = {
        "mean_tokens_per_word": round(float(np.mean(tok_lens / word_counts)), 3),
        "median_token_length": float(np.median(tok_lens)),
        "p95_token_length": float(np.percentile(tok_lens, 95)),
        "pct_truncated_at_256": round(float(100.0 * np.mean(tok_lens > 256)), 2),
        "pct_truncated_at_512": round(float(100.0 * np.mean(tok_lens > 512)), 2),
    }

with open(OUTPUTS_DIR / "tokenizer_comparison.json", "w") as f:
    json.dump(tokenizer_stats, f, indent=2)

print(json.dumps(tokenizer_stats, indent=2))

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (662 > 512). Running this sequence through the model will result in indexing errors


...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (649 > 512). Running this sequence through the model will result in indexing errors


{
  "nlpaueb/legal-bert-base-uncased": {
    "mean_tokens_per_word": 1.2,
    "median_token_length": 105.0,
    "p95_token_length": 347.14999999999986,
    "pct_truncated_at_256": 12.2,
    "pct_truncated_at_512": 1.6
  },
  "bert-base-uncased": {
    "mean_tokens_per_word": 1.257,
    "median_token_length": 109.5,
    "p95_token_length": 370.04999999999995,
    "pct_truncated_at_256": 13.8,
    "pct_truncated_at_512": 1.6
  },
  "camembert-base": {
    "mean_tokens_per_word": 1.924,
    "median_token_length": 169.0,
    "p95_token_length": 555.3499999999997,
    "pct_truncated_at_256": 28.4,
    "pct_truncated_at_512": 7.2
  }
}


In [4]:
# Step 3: Train Fast TF-IDF + Logistic Regression Baseline & Save Logits
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_texts = list(ledgar["train"]["text"])
train_labels = np.array(ledgar["train"]["label"], dtype=np.int64)
val_texts = list(ledgar["validation"]["text"])
val_labels = np.array(ledgar["validation"]["label"], dtype=np.int64)
test_texts = list(ledgar["test"]["text"])
test_labels = np.array(ledgar["test"]["label"], dtype=np.int64)

print("Fitting TF-IDF + Logistic Regression baseline on 60,000 clauses...")
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

clf = LogisticRegression(C=2.0, max_iter=300, random_state=42)
clf.fit(X_train, train_labels)

tfidf_val_logits = clf.decision_function(X_val).astype(np.float32)
tfidf_test_logits = clf.decision_function(X_test).astype(np.float32)

tfidf_val_acc = float(np.mean(np.argmax(tfidf_val_logits, axis=1) == val_labels))
tfidf_test_acc = float(np.mean(np.argmax(tfidf_test_logits, axis=1) == test_labels))
print(f"TF-IDF Baseline -> Val Accuracy: {tfidf_val_acc:.4f} | Test Accuracy: {tfidf_test_acc:.4f}")

np.save(OUTPUTS_DIR / "tfidf_val_logits.npy", tfidf_val_logits)
np.save(OUTPUTS_DIR / "tfidf_test_logits.npy", tfidf_test_logits)
np.save(OUTPUTS_DIR / "val_labels.npy", val_labels)
np.save(OUTPUTS_DIR / "test_labels.npy", test_labels)
with open(OUTPUTS_DIR / "label_names.json", "w") as f:
    json.dump(label_names, f, indent=2)

Fitting TF-IDF + Logistic Regression baseline on 60,000 clauses...
TF-IDF Baseline -> Val Accuracy: 0.8629 | Test Accuracy: 0.8641


In [5]:
# Step 4: Fine-Tune Legal-BERT on Colab GPU & Export Validation/Test Logits
import time
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 64  # Fast on Colab Pro A100/L4 (reduce to 32 if using T4)
EPOCHS = 3
LR = 3e-5
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Training on device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Pre-tokenize splits once in memory for maximum GPU utilization
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized_ledgar = ledgar.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_ledgar = tokenized_ledgar.rename_column("label", "labels")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(tokenized_ledgar["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(tokenized_ledgar["validation"], batch_size=128, shuffle=False, collate_fn=collator)
test_loader = DataLoader(tokenized_ledgar["test"], batch_size=128, shuffle=False, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def eval_loader(m, loader):
    m.eval()
    logits_list, labels_list = [], []
    total_loss, total_n = 0.0, 0
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = m(**batch)
            loss = criterion(out.logits, labels)
        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs
        logits_list.append(out.logits.detach().cpu().numpy().astype(np.float32))
        labels_list.append(labels.detach().cpu().numpy().astype(np.int64))
    return np.concatenate(logits_list), np.concatenate(labels_list), total_loss / max(1, total_n)

history = []
best_val_acc = -1.0
best_val_logits = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = model(**batch)
            loss = criterion(out.logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= scale_before:
            scheduler.step()
        bs = labels.size(0)
        running_loss += float(loss.item()) * bs
        seen += bs
        pbar.set_postfix({"train_loss": f"{running_loss / seen:.4f}"})

    val_logits, val_y, val_loss = eval_loader(model, val_loader)
    val_acc = float(np.mean(np.argmax(val_logits, axis=1) == val_y))
    rec = {
        "epoch": epoch,
        "train_loss": round(running_loss / seen, 4),
        "val_loss": round(val_loss, 4),
        "val_accuracy": round(val_acc, 4),
        "elapsed_sec": round(time.time() - t0, 1),
    }
    history.append(rec)
    print(rec)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_logits = val_logits
        model.save_pretrained("best_legal_bert")
        tokenizer.save_pretrained("best_legal_bert")

best_model = AutoModelForSequenceClassification.from_pretrained("best_legal_bert").to(device)
test_logits, test_y, test_loss = eval_loader(best_model, test_loader)
test_acc = float(np.mean(np.argmax(test_logits, axis=1) == test_y))
print(f"Final Legal-BERT Test Accuracy (Micro-F1): {test_acc:.4f} (Test Loss: {test_loss:.4f})")

np.save(OUTPUTS_DIR / "legal_bert_val_logits.npy", best_val_logits)
np.save(OUTPUTS_DIR / "legal_bert_test_logits.npy", test_logits)
with open(OUTPUTS_DIR / "training_history.json", "w") as f:
    json.dump({"model": MODEL_NAME, "test_accuracy": round(test_acc, 4), "history": history}, f, indent=2)

Training on device: cuda


Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `2`.


...

Epoch 3/3:   0%|          | 0/938 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.4321, 'val_loss': 0.522, 'val_accuracy': 0.8612, 'elapsed_sec': 822.7}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final Legal-BERT Test Accuracy (Micro-F1): 0.8598 (Test Loss: 0.5380)


In [6]:
# Step 4b: Fine-Tune Legal-BERT with LoRA (r=8, alpha=16) for 3 Epochs on T4 GPU
# Self-contained cell: can be run right after Step 1, or standalone in a fresh Colab T4 session!
import json, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 64
EPOCHS = 3
LORA_R = 8
LORA_ALPHA = 16.0
LORA_LR = 8e-4  # Standard LoRA learning rate (10x full fine-tuning lr=3e-5)
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Running LoRA (r={LORA_R}, alpha={LORA_ALPHA}) on device: {device}")

# Load dataset & tokenize if not already in memory from earlier cells
if "tokenized_ledgar" not in globals():
    ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
    label_names = ledgar["train"].features["label"].names
    num_classes = len(label_names)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenized_ledgar = ledgar.map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
        batched=True,
        remove_columns=["text"],
    ).rename_column("label", "labels")
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(tokenized_ledgar["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(tokenized_ledgar["validation"], batch_size=128, shuffle=False, collate_fn=collator)
    test_loader = DataLoader(tokenized_ledgar["test"], batch_size=128, shuffle=False, collate_fn=collator)


class LoRALinear(nn.Module):
    """Low-Rank Adaptation layer (Hu et al., 2021): h = W x + (alpha / r) * B (A x)."""

    def __init__(self, base_layer: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base_layer = base_layer
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        for p in self.base_layer.parameters():
            p.requires_grad = False
        self.lora_A = nn.Parameter(
            torch.randn(r, self.in_features, device=base_layer.weight.device, dtype=base_layer.weight.dtype) * 0.02
        )
        self.lora_B = nn.Parameter(
            torch.zeros(self.out_features, r, device=base_layer.weight.device, dtype=base_layer.weight.dtype)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base_layer(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling


def apply_lora_to_bert(model: nn.Module, r: int = 8, alpha: float = 16.0) -> nn.Module:
    for name, param in model.named_parameters():
        param.requires_grad = "classifier" in name
    for module in model.modules():
        for attr in ("query", "value"):
            if hasattr(module, attr) and isinstance(getattr(module, attr), nn.Linear):
                setattr(module, attr, LoRALinear(getattr(module, attr), r=r, alpha=alpha))
    return model


def count_parameters_and_memory(model: nn.Module) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    mem_bytes = trainable * 16 + frozen * 2
    return {
        "total_params": int(total),
        "trainable_params": int(trainable),
        "frozen_params": int(frozen),
        "trainable_pct": round(100.0 * trainable / max(1, total), 4),
        "estimated_param_grad_opt_mb": round(mem_bytes / 1e6, 2),
    }


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

lora_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes).to(device)
lora_model = apply_lora_to_bert(lora_model, r=LORA_R, alpha=LORA_ALPHA)
param_stats = count_parameters_and_memory(lora_model)
print("LoRA Parameter & Memory Accounting:", json.dumps(param_stats, indent=2))

trainable_params = [p for p in lora_model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LORA_LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


@torch.no_grad()
def eval_lora_loader(m, loader):
    m.eval()
    logits_list, labels_list = [], []
    total_loss, total_n = 0.0, 0
    for batch in loader:
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = m(**batch)
            loss = criterion(out.logits, labels)
        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs
        logits_list.append(out.logits.detach().cpu().numpy().astype(np.float32))
        labels_list.append(labels.detach().cpu().numpy().astype(np.int64))
    return np.concatenate(logits_list), np.concatenate(labels_list), total_loss / max(1, total_n)


lora_history = []
best_lora_val_acc = -1.0
best_lora_val_logits = None
best_lora_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    lora_model.train()
    running_loss, seen = 0.0, 0
    pbar = tqdm(train_loader, desc=f"LoRA Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad(set_to_none=True)
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp):
            out = lora_model(**batch)
            loss = criterion(out.logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() >= scale_before:
            scheduler.step()
        bs = labels.size(0)
        running_loss += float(loss.item()) * bs
        seen += bs
        pbar.set_postfix({"train_loss": f"{running_loss / seen:.4f}"})

    val_logits, val_y, val_loss = eval_lora_loader(lora_model, val_loader)
    val_preds = np.argmax(val_logits, axis=1)
    val_acc = float(np.mean(val_preds == val_y))
    val_macro_f1 = float(f1_score(val_y, val_preds, average="macro", zero_division=0))
    rec = {
        "epoch": epoch,
        "train_loss": round(running_loss / seen, 4),
        "val_loss": round(val_loss, 4),
        "val_accuracy": round(val_acc, 4),
        "val_macro_f1": round(val_macro_f1, 4),
        "elapsed_sec": round(time.time() - t0, 1),
    }
    lora_history.append(rec)
    print(rec)
    if val_acc > best_lora_val_acc:
        best_lora_val_acc = val_acc
        best_lora_val_logits = val_logits
        best_lora_state = {k: v.detach().cpu().clone() for k, v in lora_model.state_dict().items()}

peak_vram_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1) if torch.cuda.is_available() else None
lora_model.load_state_dict({k: v.to(device) for k, v in best_lora_state.items()})
lora_test_logits, test_y, lora_test_loss = eval_lora_loader(lora_model, test_loader)
test_preds = np.argmax(lora_test_logits, axis=1)
lora_test_acc = float(np.mean(test_preds == test_y))
lora_test_macro_f1 = float(f1_score(test_y, test_preds, average="macro", zero_division=0))
lora_test_weighted_f1 = float(f1_score(test_y, test_preds, average="weighted", zero_division=0))

np.save(OUTPUTS_DIR / "lora_val_logits.npy", best_lora_val_logits)
np.save(OUTPUTS_DIR / "lora_test_logits.npy", lora_test_logits)

lora_summary = {
    "model": MODEL_NAME,
    "method": "LoRA (query, value + classifier head)",
    "r": LORA_R,
    "alpha": LORA_ALPHA,
    "lr": LORA_LR,
    "epochs": EPOCHS,
    "param_stats": param_stats,
    "peak_vram_mb": peak_vram_mb,
    "test_accuracy": round(lora_test_acc, 4),
    "test_macro_f1": round(lora_test_macro_f1, 4),
    "test_weighted_f1": round(lora_test_weighted_f1, 4),
    "test_loss": round(lora_test_loss, 4),
    "total_elapsed_sec": round(sum(h["elapsed_sec"] for h in lora_history), 1),
    "history": lora_history,
}
with open(OUTPUTS_DIR / "lora_training_history.json", "w") as f:
    json.dump(lora_summary, f, indent=2)

print("\n=== COPY-PASTE THIS JSON SUMMARY ===")
print(json.dumps(lora_summary, indent=2))


LoRA Parameter & Memory Accounting: {
  "total_params": 109854052,
  "trainable_params": 371812,
  "frozen_params": 109482240,
  "trainable_pct": 0.3385,
  "estimated_param_grad_opt_mb": 224.91
}


Epoch 1/3 step 250/937 - train_loss=3.4538 (50.9s)


Epoch 1/3 step 500/937 - train_loss=2.2256 (90.7s)


Epoch 1/3 step 750/937 - train_loss=1.7369 (131.7s)


Epoch 1/3 step 937/937 - train_loss=1.5338 (160.9s)


...

Epoch 3/3 step 500/937 - train_loss=0.4961 (79.3s)


Epoch 3/3 step 750/937 - train_loss=0.4892 (118.7s)


Epoch 3/3 step 937/937 - train_loss=0.4886 (149.1s)


Epoch summary: {'epoch': 3, 'train_loss': 0.4886, 'val_loss': 0.525, 'val_accuracy': 0.8493, 'val_macro_f1': 0.7523, 'elapsed_sec': 159.5}


=== LORA SUMMARY JSON ===
{
  "model": "nlpaueb/legal-bert-base-uncased",
  "method": "LoRA (query, value + classifier head)",
  "r": 8,
  "alpha": 16.0,
  "lr": 0.0008,
  "epochs": 3,
  "param_stats": {
    "total_params": 109854052,
    "trainable_params": 371812,
    "frozen_params": 109482240,
    "trainable_pct": 0.3385,
    "estimated_param_grad_opt_mb": 224.91
  },
  "test_accuracy": 0.8473,
  "test_accuracy_ci": [
    0.8397,
    0.8542
  ],
  "test_macro_f1": 0.7539,
  "test_weighted_f1": 0.837,
  "test_loss": 0.5374,
  "total_elapsed_sec": 491.4,
  "history": [
    {
      "epoch": 1,
      "train_loss": 1.5338,
      "val_loss": 0.665,
      "val_accuracy": 0.8125,
      "val_macro_f1": 0.6975,
      "elapsed_sec": 173.1
    },
    {
      "epoch": 2,
      "train_loss": 0.5991,
      "val_loss": 0.5494,
      "val_accuracy": 0.8385,
      "val_macro_f1": 0.7329,
      "elapsed_sec": 158.8
    },
    {
      "epoch": 3,
      "train_loss": 0.4886,
      "val_loss": 0.525,
  

In [7]:
# Step 5: Zip and Download `outputs.zip` to Your Mac
import shutil
shutil.make_archive("outputs", "zip", root_dir=".", base_dir="outputs")
print("Created outputs.zip!")

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not running in Google Colab; outputs are saved in ./outputs/")

Created outputs.zip!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>